In [ ]:
Structure of the data file system: 

data/
 ├── serial-lt-hsc/
 │    ├── well_001/
 │    │     ├── 00d00h00m.tif
 │    │     ├── 00d03h00m.tif
 │    │     └── ...
 │    └── ...
 └── others/
      ├── well_101/
      │     ├── 00d00h00m.tif
      │     └── ...

In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import shutil
import tifffile as tiff
import cv2, os
import sys
from shapely.geometry import Polygon
import imagecodecs
from sklearn.cluster import DBSCAN
from PIL import Image

PARENTAL_DIR = "/path/to/image/folder/"
ROOT = Path(PARENTAL_DIR+"data/")

In [ ]:
_last_baseline_max_val = 0  # initial default

def get_max_if_baseline(file_path):
    global _last_baseline_max_val
    if '00d00h00m' not in os.path.basename(file_path):
        if '00d03h00m' not in os.path.basename(file_path):
            return _last_baseline_max_val
    img = tiff.imread(file_path)
    blue = img[:, :, 2]
    _last_baseline_max_val = int(np.max(blue))
    return _last_baseline_max_val

def get_contours (file_name, threshold=0.5):
    img = tiff.imread(file_name)
    # set the areas with ruler and time stamp to 0
    img[1100:1200,1:390,:] = 0
    img[1160:1200,1020:1220,:] = 0
    norm_img = np.zeros_like(img, dtype=np.float32)
    for c in range(3):
        channel = img[:,:,c]
        min_val = np.percentile(channel, 75) # min threshold can be adjusted
        max_val = get_max_if_baseline(file_name)
        norm_img[:,:,c] = (channel - min_val) / (max_val - min_val + 1e-8
    contours_list = []
    for c in [0,2]:  
        channel = norm_img[:,:,c]
        mask = (channel > threshold).astype(np.uint8)
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        print(f"channel {c}: {len(contours)}", end="; ")
        contours_list.append(contours)

    contoursA = contours_list[0]
    contoursB = contours_list[1]
    out = []
    cnt_min = 2.1 # define the minimal size of contours
    for cntA in contoursA:
        if len(cntA) < cnt_min:
            continue
        polyA = Polygon(cntA[:, 0, :]).buffer(0)
        for cntB in contoursB:
            if len(cntB) < cnt_min:
                continue
            polyB = Polygon(cntB[:, 0, :]).buffer(0)            
            inter = polyA.intersection(polyB)
            union = polyA.union(polyB)
            union_dilated = union.buffer(2)  # dilate by n pixels, can be adjusted
            if not inter.is_empty and inter.area > 0:
                if union.area <= 50: # define upper threhold of cell size
                    out.append(union_dilated)
    print(f"intersected: {len(out)}")

    new_img = np.zeros_like(img, dtype=np.uint8)
    for poly in out:
        if poly.geom_type == "MultiPolygon":
            geoms = list(poly.geoms)
        else:
            geoms = [poly]
        for g in geoms:
            if not hasattr(g, "exterior"):
                continue
            coords = np.array(g.exterior.coords, dtype=np.int32).reshape(-1, 1, 2)
            cv2.fillPoly(new_img, [coords], (255, 255, 255))
    path = os.path.dirname(file_name)
    if not os.path.exists(f'{path}/masks'):
        os.makedirs(f'{path}/masks')
    name_no_ext = os.path.splitext(os.path.basename(file_name))[0]
    cv2.imwrite(f"{path}/masks/{name_no_ext}_cv2_masks.png", new_img)
    return out

# for each tif file in the folders, run get_contours, recursively
def run_on_all_tifs(root):
    for tif in sorted(root.rglob("*.tif*")):  
        try:
            print(tif, end=" ", flush=True)
            get_contours(str(tif), threshold=0.2)    #threshold for brightness of finding contour
        except Exception as e:
            print(f"\nError processing {tif}: {e}", file=sys.stderr)
run_on_all_tifs(ROOT)

def extract_components_full(mask):
    #finds connected components in a binary mask and returns metadata for each component
    num, labels = cv2.connectedComponents((mask>0).astype(np.uint8))
    comps = []
    for lab in range(1, num):
        comp_mask = (labels==lab).astype(np.uint8)
        ys, xs = np.where(comp_mask)
        if ys.size==0:
            continue
        area = ys.size
        cx = xs.mean(); cy = ys.mean()
        x0, y0, x1, y1 = xs.min(), ys.min(), xs.max(), ys.max()
        full_mask = np.zeros_like(mask, dtype=np.uint8)
        full_mask[ys, xs] = 1
        comps.append({'mask': full_mask, 'area': area, 'centroid': (cx, cy), 'bbox': (x0,y0,x1,y1)})
    return comps

def load_masks(folder, suffix='_cv2_masks.png'):
    files = sorted([f for f in os.listdir(folder) if f.endswith(suffix)], key=lambda s: s.lower())
    frames = []
    for f in files:
        m = cv2.imread(os.path.join(folder,f), cv2.IMREAD_GRAYSCALE)
        if m is None:
            continue
        _, m = cv2.threshold(m,127,255,cv2.THRESH_BINARY)
        frames.append((f, m))
    return frames

def consensus_mask_from_frames(frames, presence_frac=0.7, cluster_multiplier=10, check_first_n=12):
    if len(frames) == 0:
        return None
    filenames, masks = zip(*frames)
    H, W = masks[0].shape
    n_frames = len(masks)
    n_check = min(check_first_n, n_frames)  # use first 12 (or fewer)
    all_comps = []
    centroids = []
    meta = []
    for i, mask in enumerate(masks):
        comps = extract_components_full(mask)
        all_comps.append(comps)
        for j, c in enumerate(comps):
            centroids.append(c['centroid'])
            meta.append((i, j))
    if len(centroids) == 0:
        return np.zeros((H, W), dtype=np.uint8)
    median_radius = np.median([max(1, np.sqrt(c['area'])) for comps in all_comps for c in comps])
    eps = cluster_multiplier * median_radius
    # cluster_multiplier scales the DBSCAN clustering radius (eps).
    clustering = DBSCAN(eps=eps, min_samples=10).fit(centroids)
    labels = clustering.labels_
    final_mask = np.zeros((H, W), dtype=np.uint8)
    first_check_frames = set(range(n_check))
    for cluster_id in np.unique(labels):
        members = [k for k, lab in enumerate(labels) if lab == cluster_id]
        frames_present = set(meta[k][0] for k in members)
        frames_present_in_check = len(frames_present & first_check_frames)
        presence_fraction = frames_present_in_check / n_check
        if presence_fraction < presence_frac:
            continue
        count_map = np.zeros((H, W), dtype=np.uint16)
        for k in members:
            fi, ci = meta[k]
            comp = all_comps[fi][ci]
            count_map += comp['mask'].astype(np.uint16)
        thr = int(np.ceil(presence_frac * n_check))  # require presence in >=70% of first n_check frames
        consensus = (count_map >= thr).astype(np.uint8) * 255
        final_mask = cv2.bitwise_or(final_mask, consensus)
    return final_mask

def process_series_folder(folder, out_name='final_consensus_mask.png', suffix='_cv2_masks.png',
                          presence_frac=0.1, cluster_multiplier=10):
    frames = load_masks(folder, suffix)
    if not frames:
        return
    final = consensus_mask_from_frames(frames, presence_frac, cluster_multiplier)
    if final is None:
        return
    out_path = os.path.join(folder, out_name)
    cv2.imwrite(out_path, final)
    
def largest_cluster_circle_overlay(mask_path, out_path,
                                   bin_thresh=127, dbscan_eps=50, dbscan_min_samples=1,
                                   circle_color=(0,0,255),  # BGR red
                                   circle_thickness=2
                                   ):
    m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if m is not None:
        _, binm = cv2.threshold(m, bin_thresh, 255, cv2.THRESH_BINARY)
        base_rgb = cv2.cvtColor(m, cv2.COLOR_GRAY2BGR)
        num, labels = cv2.connectedComponents((binm > 0).astype(np.uint8))
        centroids = []
        areas = []
        for lab in range(1, num):
            ys, xs = np.where(labels == lab)
            if ys.size == 0:
                continue
            cx = xs.mean(); cy = ys.mean()
            centroids.append((cx, cy))
            areas.append(xs.size)  
        if not centroids:
            cv2.imwrite(out_path, base_rgb)
            return
        # cluster centroids with DBSCAN
        X = np.array(centroids)
        clustering = DBSCAN(eps=dbscan_eps, min_samples=dbscan_min_samples).fit(X)
        labels_cl = clustering.labels_
        unique_labels = np.unique(labels_cl)
        best_label = None
        best_area = -1
        for ul in unique_labels:
            members = np.where(labels_cl == ul)[0]
            total_area = sum(areas[i] for i in members)
            if total_area > best_area:
                best_area = total_area
                best_label = ul
        largest_pts = X[labels_cl == best_label]
        mask_coords = []
        member_labels = [lab for idx, lab in enumerate(range(1, num)) if idx in np.where(labels_cl == best_label)[0]]
        for comp_lab in member_labels:
            ys, xs = np.where(labels == comp_lab)
            if ys.size:
                pts = np.column_stack((xs, ys))  
                mask_coords.append(pts)
        if mask_coords:
            pts_all = np.vstack(mask_coords).astype(np.float32)
            (cx, cy), radius = cv2.minEnclosingCircle(pts_all)
        else:
            (cx, cy), radius = cv2.minEnclosingCircle(largest_pts.astype(np.float32))

        center = (int(round(cx)), int(round(cy)))
        radius = int(round(radius) * 3)  #relax the circle to ensure all cells are included
        cv2.circle(base_rgb, center, radius, circle_color, thickness=circle_thickness, lineType=cv2.LINE_AA)
        cv2.imwrite(out_path, base_rgb)

def crop_tiffs_to_red_circle(mask_path, src_folder, out_folder, red_thresh=200, pad=False):
    mask_path = Path(mask_path)
    src_folder = Path(src_folder)
    out_folder = Path(out_folder)
    out_folder.mkdir(parents=True, exist_ok=True)

    if not mask_path.exists():
        raise FileNotFoundError(f"Mask not found: {mask_path}")
    mask = Image.open(mask_path).convert("RGB")
    mask_np = np.array(mask)
    r = mask_np[:, :, 0].astype(int)
    g = mask_np[:, :, 1].astype(int)
    b = mask_np[:, :, 2].astype(int)
    red_mask = (r >= red_thresh) & (g <= max(0, red_thresh - 120)) & (b <= max(0, red_thresh - 120))
    if not red_mask.any():
        # try a looser heuristic before failing
        red_mask = (r >= red_thresh) & (r >= g + 50) & (r >= b + 50)
        if not red_mask.any():
            print("No red pixels found in mask. Check mask_path and thresholds.")
    if red_mask.any():
        ys, xs = np.where(red_mask)
        left, upper = int(xs.min()), int(ys.min())
        right, lower = int(xs.max()) + 1, int(ys.max()) + 1
        bbox = (left, upper, right, lower)

        # Process TIFFs
        for tiff_path in sorted(src_folder.glob("*.tif*")):
            try:
                with Image.open(tiff_path) as img:
                    w, h = img.size
                    if left < 0 or upper < 0 or right > w or lower > h:
                        if not pad:
                            print(f"Skipping {tiff_path.name}: bbox {bbox} outside image bounds {img.size}")
                            continue
                        crop_w, crop_h = right - left, lower - upper
                        paste_x = max(0, -left)
                        paste_y = max(0, -upper)
                        canvas = Image.new(img.mode, (crop_w, crop_h))
                        src_left = max(0, left)
                        src_upper = max(0, upper)
                        src_right = min(w, right)
                        src_lower = min(h, lower)
                        src_box = (src_left, src_upper, src_right, src_lower)
                        region = img.crop(src_box)
                        canvas.paste(region, (src_left - left, src_upper - upper))
                        cropped = canvas
                    else:
                        cropped = img.crop(bbox)

                    out_path = out_folder / tiff_path.name
                    if out_path.exists():
                        out_path.unlink()
                    cropped.save(out_path, format="TIFF")
            except Exception as e:
                print(f"Failed {tiff_path.name}: {e}")
        return bbox

def process_target_root(root_name: str) -> None:
    target_dir = Path(PARENTAL_DIR) / "data" / root_name
    if not target_dir.exists():
        return

    for sub in sorted(p for p in target_dir.iterdir() if p.is_dir()):
        if "masked_cropped" in sub.name or "masks" in sub.name:
            continue

        crop_tiffs_to_red_circle(
            sub / "1masks" / "largest_cluster_circle.png",
            sub,
            sub / "cropped_tiffs",
            red_thresh=200,
            pad=False,
        )

In [ ]:
for name in sorted(os.listdir(ROOT)):
    sub = os.path.join(ROOT, name)
    if not os.path.isdir(sub):
        continue
    for dirpath, dirnames, filenames in os.walk(sub):
        process_series_folder(dirpath)
        largest_cluster_circle_overlay(dirpath+'/masks/final_consensus_mask.png', dirpath+'/masks/largest_cluster_circle.png',
                            bin_thresh=127, dbscan_eps=100, dbscan_min_samples=1)

In [ ]:

for root_name in ("serial-lt-hsc", "others"):
    process_target_root(root_name)